# 心思维｜检查已保存的 TorchCREPE 音高文件

本 Notebook **只读取已有的 `vocal_pitch_raw.npz` 文件**，检查数组名称、形状、数据类型和数值范围，并保存一份 JSON 检查报告。不会运行 CREPE、不会下载模型、不会占用 GPU。

请在 Colab 打开后只运行下面一个代码单元。Google Drive 会要求授权。

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np
import json

drive.mount('/content/drive')
npz_path = Path('/content/drive/MyDrive/XinsW_Lead_Transcription/TorchCREPE_Test/vocal_pitch_raw.npz')
if not npz_path.is_file():
    folder = npz_path.parent
    found = list(folder.glob('*.npz')) if folder.exists() else []
    raise FileNotFoundError(f'未找到：{npz_path}\n该目录下找到的 NPZ 文件：{[p.name for p in found]}')

report = {'file': npz_path.name, 'path': str(npz_path), 'arrays': {},
          'notes': ['只检查已有数据，不运行模型。', '未启用 pickle；对象数组不会被反序列化。']}
with np.load(npz_path, allow_pickle=False) as archive:
    print('文件大小：', f'{npz_path.stat().st_size:,}', 'bytes')
    print('数组数量：', len(archive.files))
    for key in archive.files:
        try:
            arr = archive[key]
        except ValueError as exc:
            report['arrays'][key] = {'readable': False, 'reason': str(exc)}
            print(f'\\n[{key}] 无法安全读取：{exc}')
            continue
        info = {'shape': list(arr.shape), 'dtype': str(arr.dtype), 'size': int(arr.size)}
        if arr.size and np.issubdtype(arr.dtype, np.number):
            flat = arr.reshape(-1)
            finite = np.isfinite(flat)
            info['finite_count'] = int(finite.sum())
            if finite.any():
                vals = flat[finite]
                info['min'] = float(vals.min())
                info['max'] = float(vals.max())
                info['sample'] = [float(x) for x in vals[:12]]
            else:
                info['sample'] = []
        else:
            info['sample'] = [str(x) for x in arr.reshape(-1)[:12]] if arr.size else []
        report['arrays'][key] = info
        print(f'\\n[{key}]')
        print(json.dumps(info, ensure_ascii=False, indent=2))

report_path = npz_path.with_name('vocal_pitch_raw_inspection.json')
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('\\n检查完成，报告已保存：', report_path)
print('请把上面输出或 vocal_pitch_raw_inspection.json 发回对话；不需要上传原始音频，也不需要重新运行模型。')
